# visualise

> Create figures

In [ ]:
# | default_exp euclid.visualise

In [ ]:
# | exporti

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.nddata import CCDData
from astropy.visualization import simple_norm
from matplotlib.patches import Ellipse

from nicl.autoprof import get_autoprof_info
from nicl.main import configure_logging
from nicl.utilities import get_pixel_scale

In [ ]:
# | hide

from nicl.euclid.data_access import default_data_path

In [ ]:
# | exporti

plt.style.use("nicl.euclid.v1nicl")

In [ ]:
configure_logging(name="__main__", level="DEBUG")

<Logger __main__ (DEBUG)>

In [ ]:
# | export


def plot_cluster_annuli(
    image_filename,
    profile_filename=None,
    mask_filename=None,
    true_profile_filename=None,
    ax=None,
    color="red",
    true_color="blue",
):
    if ax is None:
        fig, ax = plt.subplots()
    cmap = plt.cm.grey
    cmap.set_bad("black", alpha=1)
    image = CCDData.read(image_filename, unit="adu")
    pixelscale = get_pixel_scale(image).to_value("arcsec")
    if mask_filename is not None:
        mask = CCDData.read(mask_filename, unit="adu")
        image.data[mask.data > 0] = np.nan
    info = get_autoprof_info(profile_filename)
    x, y = info["centre"]
    central_adu = 10 ** (0.4 * (info["zeropoint"] - info["central_sb"])) * pixelscale**2
    r_fit = info["fit_limit"]
    r_min = r_fit * 0.01
    r_max = r_fit * 3
    vmin = -3 * info["noise"]
    vmax = central_adu / 10
    asinh_a = (3 * info["noise"] - vmin) / (vmax - vmin)
    norm = simple_norm(image.data, "asinh", asinh_a=asinh_a, vmin=vmin, vmax=vmax)
    xh = (image.shape[1] - 1) / 2
    yh = (image.shape[0] - 1) / 2
    # set the axes to be centred of the ellipses and in units of arcsec
    extent = np.array([x - 2 * xh - 0.5, x - 0.5, y - 2 * yh - 0.5, y - 0.5])
    extent *= pixelscale
    ax.set_xlim(-2 * r_fit, 2 * r_fit)
    ax.set_ylim(-2 * r_fit, 2 * r_fit)
    ax.imshow(
        image.data,
        origin="lower",
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
        extent=extent,
    )
    ax.set_xlabel(r"$\Delta RA$ [arcsec]")
    ax.set_ylabel(r"$\Delta Dec$ [arcsec]")
    profile = pd.read_csv(profile_filename, skiprows=1)
    rad = profile["R"].values
    ellip = profile["ellip"].values
    pa = profile["pa"].values - 90
    if true_profile_filename is not None:
        true_profile = pd.read_csv(true_profile_filename, skiprows=1)
        true_rad = true_profile["R"].values
        true_ellip = true_profile["ellip"].values
        true_pa = true_profile["pa"].values - 90
    for i in range(len(rad) - 1):
        r_outer = rad[i + 1]
        if r_outer > r_min and r_outer < r_max:
            if r_outer <= r_fit:
                ls = "-"
            else:
                ls = "--"
            ellipse = Ellipse(
                (0, 0),
                width=2 * r_outer,
                height=2 * r_outer * (1 - ellip[i]),
                angle=pa[i],
                edgecolor=color,
                facecolor="none",
                ls=ls,
                lw=0.5,
            )
            ax.add_patch(ellipse)
            if true_profile_filename is not None:
                true_ellipse = Ellipse(
                    (0, 0),
                    width=2 * true_rad[i],
                    height=2 * true_rad[i] * (1 - true_ellip[i]),
                    angle=true_pa[i],
                    edgecolor=true_color,
                    facecolor="none",
                    ls=":",
                    lw=0.5,
                )
            ax.add_patch(true_ellipse)

In [ ]:
name = "basic_test"
cluster_id = "cluster"
measure_path = default_data_path("test_measure") / name / cluster_id
image_path = default_data_path("test_image") / name
image_path = measure_path
band = "YJH"
mask_filter = "YJH"
image_filename = image_path / f"{cluster_id}_{band}.fits"
mask_filename = measure_path / f"cluster_{mask_filter}_measurement_mask.fits"
profile_filename = (
    measure_path / f"autoprof_results/cluster-mask_{mask_filter}-iso_{band}.prof"
)
true_profile_filename = (
    measure_path / f"autoprof_results/cluster-no_mask-iso_{band}_no_noise.prof"
)
plot_cluster_annuli(
    image_filename, profile_filename, mask_filename, true_profile_filename
)